# 策略概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

**HDBSCAN Cluster SSD-DTW-PCA PCA5 Resid** 在 PCA5 版聚類分組之前，先對報酬做**因子殘差化**（移除市場因子與產業因子），改以**特殊性報酬（idiosyncratic returns）**建構 5 維 PCA 因子載荷，再執行聚類：

1. 日報酬 → **殘差化**：逐股回歸市場因子取殘差，再逐股回歸所屬產業因子取殘差
2. 對殘差報酬做 PCA，取前 5 主成分的因子載荷作為聚類座標
3. HDBSCAN 密度聚類分組（噪音點排除）
4. 群內雙向 OLS + ADF/半衰期/Hurst 篩選 → SSD/DTW 距離 → PC1 融合排序，取前 `top_n`
5. 交易期以 `ignore_ols_alpha=True` 於標準化空間重建 spread



## 為何在聚類前做因子殘差化

直接以原始報酬分群/算相關，結構會被**市場 β 齊漲齊跌**主導：多數股票對在多頭時同向、空頭時同向，
相關矩陣量到的是「大盤共動」而非「配對專屬的均衡關係」，且此結構隨市場 regime 漂移、出樣本不穩。

移除共同因子後保留的**特殊性報酬**才是配對交易的獲利來源——兩股在剝除市場與產業影響後仍持續共動，
才有結構性的均值回歸基礎，且較耐 regime 變動。


# 參考文獻與引用對應


## 文獻 1：Avellaneda & Lee (2010)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

**參考部分**：

- 以 PCA 從報酬萃取共同風險因子（eigenportfolios），將股票報酬分解為「**因子暴露 + 特殊性報酬**」
- 統計套利訊號建立在**殘差（特殊性）報酬**的均值回歸上，而非原始報酬

**為何參考**：

- 本策略階段 1 的殘差化（移除市場＋產業因子、保留特殊性報酬）即此文的核心方法；
  以殘差報酬的因子載荷作為聚類座標，使分組反映「特殊性共動」結構
- 因子載荷以 $\sqrt{\text{特徵值}}$ 加權亦沿用其 eigenportfolio 權重尺度化



## 文獻 2：Campello, Moulavi & Sander (2013)／許鈞翔 (2025)

> Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD 2013*.
> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。元智大學碩士論文。

**參考部分與理由**：

- Campello et al.：HDBSCAN 密度聚類（自動群數、噪音標記）——階段 2 分組演算法
- 許鈞翔 (2025)：群內「共整合篩選 → SSD/DTW 距離 → PC1 融合排序」流程，經 DTW 距離排序流程 複用
  


# 各階段行為


## 階段 1：因子殘差化（依據：Avellaneda & Lee 2010）

對形成窗日報酬矩陣 $R \in \mathbb{R}^{(T-1)\times N}$ 依序移除兩層因子：

**市場因子**（橫斷面平均報酬）：

$$f^{mkt}_t = \frac{1}{N}\sum_{i} R_{t,i}, \qquad
e^{(1)}_{t,i} = R_{t,i} - \hat\beta_i\, f^{mkt}_t \ \text{（逐股對 } [1, f^{mkt}] \text{ OLS 取殘差）}$$

**產業因子**（同產業市場殘差的橫斷面平均）：

$$g^{(s)}_t = \frac{1}{|s|}\sum_{i \in s} e^{(1)}_{t,i}, \qquad
e^{(2)}_{t,i} = e^{(1)}_{t,i} - \hat\gamma_i\, g^{(s_i)}_t$$

- 產業成員 < 3 檔者跳過產業層（2 檔時產業因子與殘差完全共線，會使後續 PCA 的 SVD 退化不收斂）
- 輸出 $e^{(2)}$ 每欄均值為 0，正好是 PCA 需要的去均值輸入


## 階段 2：PCA5 因子載荷 + HDBSCAN 聚類

對**殘差報酬** $e^{(2)}$ 做 PCA，取前 5 主成分：

$$\text{loadings}_i = \text{components}_{:,i} \times \sqrt{\text{explained variance}} \in \mathbb{R}^{5}$$

以 $N\times 5$ loadings 矩陣執行 HDBSCAN（`min_cluster_size=5`、`min_samples=2`、euclidean，`reduce_method="none"`）；
噪音點（`label=-1`）映射為 `"Unknown"` 排除。


## 階段 3–5：群內篩選、距離、排序（與 PCA5 版相同）

以群集標籤作為分組，複用 DTW 距離排序流程 流程：

| 步驟 | 內容 |
| :--- | :--- |
| 雙向 OLS + ADF | 兩方向各檢定，取 p 值較小者（$p < 0.01$） |
| OU 半衰期 | $\lambda < 0$ 且 $1 \le HL \le 42$ 日 |
| Hurst | $H < 0.50$ |
| 距離 | SSD 與 Sakoe-Chiba DTW（$W=15$） |
| 排序 | SSD/DTW 標準化 → PCA 取 PC1 分數升序，取前 `top_n` |

輸出欄位含 `Sector`（群集標籤）與 `Sector_A/B`（真實 GICS 回填）；
`Hedge_Ratio`、`OLS_Alpha`、`Spread_Mean/Std`、`Log_Mean/Std_A/B` 供交易期使用。


## 階段 6：交易期參數使用

`ignore_ols_alpha=True`，交易期於標準化空間重建 spread：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \quad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio}\cdot P'_{B,t}, \quad
Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

形成期統計量整個交易期凍結（無前視）。交易決策見 `trading/zscore_trading.ipynb`。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗 $F$ / 滾動步長 | 252 / 21 交易日 | — |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 每期配對數 |
| `factor_residual` | **True** | 聚類前移除市場＋產業因子 |
| `pca_n_components` | 5 | 殘差報酬 PCA 因子數 |
| `hdbscan_min_cluster_size` / `min_samples` | 5 / 2 | HDBSCAN 分組 |
| `method` | `ssd_dtw_pca` | PC1 融合排序 |
| `adf_pvalue_threshold` | 0.01 | ADF 顯著水準 |
| 半衰期 / Hurst | $[1,42]$ 日 / $<0.5$ | 均值回歸過濾 |
| `ignore_ols_alpha` | True | 標準化空間 spread |
